# S6E10 — 04 LightGBM + Optuna on `X_final`

> **Pipeline position:** 03 final features → **04 tuned LightGBM** → 05 CatBoost/XGB → … → 11 stack

| | |
|---|---|
| **Purpose** | Find good LightGBM hyper-parameters with Optuna and produce OOF/test predictions of the best one for the stack. |
| **Input** | `data/raw/*` (incl. `orig_data.csv`). `X_final` is rebuilt inside (same code as `03`) so the notebook is self-contained on Kaggle. |
| **Output** | `oof_lgbm_tuned_final.npy`, `test_lgbm_tuned_final.npy`, `submission_lgbm_tuned_final.csv`, `optuna_trials.csv`, `best_params.json` |
| **Runtime** | Kaggle, in the background (up to 4 h search + final 5-fold). Run once with `SMOKE = True` first (~2 min). |

**Top to bottom:**
1. Rebuild `X_final` (~5 min).
2. Optuna search (3-fold, in-fold target encoding). Every finished trial is printed and written to disk, so nothing is lost if the session dies.
3. Retrain the best parameters on the usual 5 folds.

**How to run on Kaggle:** add the competition and the dataset `deonissx/s6e10-orig-data` as inputs → *Save Version → Save & Run All (Commit)*. It runs in the background.

**Reference to beat:** tuned LGBM on `X_final`, CV 0.96072 (LB 0.96036).

> **Glossary:** *Optuna* tries parameter sets, learns which regions work (TPE sampler) and proposes the next one. *Trial* = one parameter set. *Early stopping* = stop adding trees when validation AUC stops improving.

## 1. Setup
Paths for Kaggle/local and the shared 5-fold split (`random_state=42`). Same code as in `03`.

In [ ]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e10') if Path('/kaggle/input/competitions/playground-series-s6e10').exists() \
           else next((p for p in (Path('data/raw'), Path('../data/raw')) if p.exists()), Path('data/raw'))
ORIG_PATH = next((p for p in (Path('/kaggle/input/datasets/deonissx/s6e10-orig-data/orig_data.csv'), Path('data/raw/orig_data.csv'), Path('../data/raw/orig_data.csv')) if p.exists()), Path('data/raw/orig_data.csv'))
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else next((p for p in (Path('data/processed'), Path('../data/processed')) if p.exists()), Path('data/processed'))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same 5 folds (seed 42) as every other notebook, so OOF predictions can be stacked.
N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print('data:', DATA_DIR, '| orig exists:', ORIG_PATH.exists(), '| out:', OUT_DIR)

## 2. Load data, preprocessing, frequency features
Identical to `03_final_features`: fill `arrival_delay` gaps from `departure_delay` (+ missing flag), add two label-free frequency features. See `03` for the reasoning.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

# 1. missing-value flag BEFORE fillna
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)
# 2. fill gaps in arrival_delay with departure_delay
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(train['departure_delay_in_minutes'])
test['arrival_delay_in_minutes']  = test['arrival_delay_in_minutes'].fillna(test['departure_delay_in_minutes'])
# 3. target
y_train = train['satisfaction'].astype(int)
# 4. drop id / target / arrival_delay
CAT_COLS  = ['gender', 'customer_type', 'type_of_travel', 'class']
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']
X_raw      = train.drop(columns=DROP_COLS)
X_test_raw = test.drop(columns=['id', 'arrival_delay_in_minutes'])

def add_features(train_df, test_df):
    """flight_distance_freq and flight_distance_type_of_travel_freq (counts over train+test, no target)."""
    train_df = train_df.copy(); test_df = test_df.copy()
    full = pd.concat([train_df, test_df], ignore_index=True); n = len(train_df)
    counts = full['flight_distance'].map(full['flight_distance'].value_counts())
    train_df['flight_distance_freq'] = counts.iloc[:n].values
    test_df['flight_distance_freq']  = counts.iloc[n:].values
    key = full['flight_distance'].astype(str) + '_' + full['type_of_travel'].astype(str)
    counts = key.map(key.value_counts())
    train_df['flight_distance_type_of_travel_freq'] = counts.iloc[:n].values
    test_df['flight_distance_type_of_travel_freq']  = counts.iloc[n:].values
    return train_df, test_df

X_train, X_test = add_features(X_raw, X_test_raw)
for col in CAT_COLS:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')
print(f'Train: {X_train.shape}, Test: {X_test.shape} | positive rate {y_train.mean():.3f}')

## 3. Original-dataset signal (`orig_prob`, `orig_logit`)
A regularized LightGBM trained only on the original dataset scores the competition rows (no competition labels → no leakage). Explained in `03`.

In [ ]:
orig = pd.read_csv(ORIG_PATH)
orig.columns = orig.columns.str.lower().str.replace(' ', '_')
feats = [c for c in train.columns if c in orig.columns and c not in ('id', 'satisfaction', 'arrival_delay_missing')]
print(orig.shape, '| positive rate:', round(orig['satisfaction'].astype(int).mean(), 3), '| features:', len(feats))

orig_f = orig.copy()
orig_f['arrival_delay_in_minutes'] = orig_f['arrival_delay_in_minutes'].fillna(orig_f['departure_delay_in_minutes'])
orig_y = orig_f['satisfaction'].astype(int)

orig_X  = orig_f[feats].copy()
tr_orig = train[feats].copy()
te_orig = test[feats].copy()
for col in CAT_COLS:
    cats = sorted(set(orig_X[col].astype(str)) | set(tr_orig[col].astype(str)) | set(te_orig[col].astype(str)))
    for df in (orig_X, tr_orig, te_orig):
        df[col] = pd.Categorical(df[col].astype(str), categories=cats)

m_orig = lgb.LGBMClassifier(
    objective='binary', n_estimators=600, learning_rate=0.05, max_depth=8,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.5,
    reg_lambda=5.0, reg_alpha=0.05, random_state=123, n_jobs=-1, verbosity=-1)
m_orig.fit(orig_X, orig_y, categorical_feature=CAT_COLS)

p_tr = m_orig.predict_proba(tr_orig)[:, 1]
p_te = m_orig.predict_proba(te_orig)[:, 1]
orig_auc = roc_auc_score(y_train, p_tr)
print('AUC of the orig-model on competition train:', round(orig_auc, 5), '(reference 0.95472)')
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1 - 1e-6))

## 4. Route profile (`fdm_*`)
`flight_distance` ≈ route ID; each row gets the mean of all other features over its route. Explained in `03`.

In [ ]:
full = pd.concat([X_train, X_test], ignore_index=True)
n = len(X_train)
prof_cols = [c for c in X_train.columns
             if c not in ('flight_distance', 'flight_distance_freq', 'flight_distance_type_of_travel_freq')]
coded = full[prof_cols].copy()
for c in CAT_COLS:
    coded[c] = coded[c].astype('category').cat.codes
coded['flight_distance'] = full['flight_distance'].values
grp = coded.groupby('flight_distance')
PROF = pd.DataFrame({f'fdm_{c}': grp[c].transform('mean').astype('float32') for c in prof_cols})
print('profile columns:', PROF.shape[1])

## 5. Assemble `X_final` and save
Sanity checks (columns, row counts, NaN, orig-model AUC) then parquet. If an assertion fails, the features no longer match the frozen set.

In [ ]:
X_final = X_train.copy()
X_test_final = X_test.copy()
for c in PROF.columns:
    X_final[c]      = PROF[c].values[:n]
    X_test_final[c] = PROF[c].values[n:]
X_final['orig_prob'], X_final['orig_logit']           = p_tr, logit(p_tr)
X_test_final['orig_prob'], X_test_final['orig_logit'] = p_te, logit(p_te)
for col in CAT_COLS:
    X_final[col]      = X_final[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

# --- sanity checks ---
assert list(X_final.columns) == list(X_test_final.columns)
assert X_final.shape[0] == 699635 and X_test_final.shape[0] == 299844, 'unexpected row counts'
assert not X_final.isna().any().any() and not X_test_final.isna().any().any(), 'NaN in features'
assert abs(orig_auc - 0.95472) < 0.0010, f'orig-model AUC {orig_auc:.5f} differs from reference 0.95472'
print('X_final:', X_final.shape, '| X_test_final:', X_test_final.shape)

X_final.to_parquet(OUT_DIR / 'X_final_train.parquet')
X_test_final.to_parquet(OUT_DIR / 'X_final_test.parquet')
y_train.reset_index(drop=True).to_frame('y').to_parquet(OUT_DIR / 'y_train.parquet')
print('saved to', OUT_DIR, '->', sorted(p.name for p in OUT_DIR.glob('*.parquet')))

## 6. Optuna — LightGBM
**Design decisions**
- **3-fold inside the search, 5-fold for the final run.** A trial costs 3 fits instead of 5; we only need the *ranking* of parameter sets.
- **Target encoding computed once per fold, not per trial.** The data does not change between trials, so recomputing would waste time.
- **`subsample_freq=1` is required.** In LightGBM `subsample` is silently ignored unless `subsample_freq > 0` — an easy bug that makes a tuned parameter do nothing.
- **First trial = our previous best point**, enqueued as a reference inside the same setup, so the search can only improve on it.
- **Resumable:** SQLite storage + a CSV/JSON dump after every trial.
- **Search space:** learning rate 0.008–0.03 (log), `num_leaves` 64–400, `min_child_samples` 10–150, subsample/colsample, L1/L2 regularisation (log).

In [ ]:
import json, optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

# Always run once with SMOKE=True: it checks the whole code path in ~2 minutes before the multi-hour search.
SMOKE    = False      # True: 30k rows, 3 trials, 100 trees -> only checks that the code runs
N_TRIALS = 40
TIMEOUT  = 4 * 3600   # seconds; the search stops by itself, trials done so far are kept
RUN_FINAL_5FOLD = True

optuna.logging.set_verbosity(optuna.logging.INFO)     # one log line per finished trial

X_all, X_te_all, y_all = X_final.reset_index(drop=True), X_test_final.reset_index(drop=True), y_train.reset_index(drop=True)
if SMOKE:
    idx = np.random.RandomState(0).choice(len(X_all), 30000, replace=False)
    X_all, y_all = X_all.iloc[idx].reset_index(drop=True), y_all.iloc[idx].reset_index(drop=True)
    N_TRIALS, TIMEOUT = 3, 600
MAX_TREES = 100 if SMOKE else 3000

def add_te(X_fit, y_fit, other, smooth='auto'):
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=RANDOM_STATE)
    key = lambda d: d[['flight_distance']].astype(str)
    X_fit = X_fit.copy(); X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]
    out = []
    for f in other:
        f = f.copy(); f['te_flight_distance'] = enc.transform(key(f))[:, 0]; out.append(f)
    return X_fit, out

# TE is computed ONCE per fold here (not inside every trial) -> much faster
skf3 = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)
# Why 3 folds in the search but 5 in the final run: a trial costs 3 model fits instead of 5, so more trials fit in the time budget. Ranking of parameter sets is what matters here, not the exact score.
folds3 = []
for tr, va in skf3.split(X_all, y_all):
    Xtr, [Xva] = add_te(X_all.iloc[tr], y_all.iloc[tr], [X_all.iloc[va]])
    folds3.append((Xtr, y_all.iloc[tr], Xva, y_all.iloc[va]))
print('3-fold data ready:', [f[0].shape for f in folds3])

# One Optuna trial = one parameter set, scored by mean AUC over the 3 folds.
def objective(trial):
    params = dict(objective='binary', metric='auc', verbose=-1, random_state=RANDOM_STATE, n_jobs=-1,
        n_estimators=MAX_TREES, early_stopping_rounds=100,
        learning_rate=trial.suggest_float('learning_rate', 0.008, 0.03, log=True),
        num_leaves=trial.suggest_int('num_leaves', 64, 400),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 150),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        subsample_freq=1,                                  # without it `subsample` is silently ignored
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 0.9),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10, log=True))
    aucs = []
    for Xtr, ytr, Xva, yva in folds3:
        m = LGBMClassifier(**params).fit(Xtr, ytr, eval_set=[(Xva, yva)], categorical_feature=CAT_COLS)
        aucs.append(roc_auc_score(yva, m.predict_proba(Xva)[:, 1]))
    trial.set_user_attr('fold_aucs', [round(a, 5) for a in aucs])
    trial.set_user_attr('best_iter_last_fold', int(m.best_iteration_ or 0))
    return float(np.mean(aucs))

def save_progress(study, trial):
    study.trials_dataframe().to_csv(OUT_DIR / 'optuna_trials.csv', index=False)
    json.dump({'best_value': study.best_value, 'best_params': study.best_params},
              open(OUT_DIR / 'best_params.json', 'w'), indent=2)
    print(f'>>> trial {trial.number}: {trial.value:.5f} | folds {trial.user_attrs.get("fold_aucs")} | '
          f'BEST so far {study.best_value:.5f} (trial {study.best_trial.number})', flush=True)

# SQLite storage: if the session dies, rerun the cell and the study continues where it stopped.
study = optuna.create_study(direction='maximize', study_name='lgbm_final',
                            storage=f'sqlite:///{OUT_DIR}/optuna_lgbm.db', load_if_exists=True)
if len(study.trials) == 0:     # first trial = our previous best point, as a reference inside the same setup
    study.enqueue_trial({'learning_rate': 0.0116, 'num_leaves': 232, 'min_child_samples': 69,
                         'subsample': 0.985, 'colsample_bytree': 0.65, 'reg_alpha': 1.56, 'reg_lambda': 1.77})
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT, callbacks=[save_progress])
print('\nBEST:', study.best_value); print(json.dumps(study.best_params, indent=2))

## 7. Final 5-fold run with the best parameters
Same 5 folds as every other model (`random_state=42`), in-fold target encoding. Saves OOF/test predictions for the stack and a submission file. Compare `CV AUC` with the previous **0.96072**.

In [ ]:
if RUN_FINAL_5FOLD:
    # Re-train the best parameters on the usual 5 folds (more training data per fold than in the 3-fold search).
    best = dict(objective='binary', metric='auc', verbose=-1, random_state=RANDOM_STATE, n_jobs=-1,
                n_estimators=MAX_TREES, early_stopping_rounds=100, subsample_freq=1, **study.best_params)
    skf5 = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(skf5.split(X_all, y_all)); splits = splits[:1] if SMOKE else splits
    oof = np.full(len(X_all), np.nan); test_pred = np.zeros(len(X_te_all)); aucs = []
    for k, (tr, va) in enumerate(splits, 1):
        Xtr, [Xva, Xte] = add_te(X_all.iloc[tr], y_all.iloc[tr], [X_all.iloc[va], X_te_all])
        m = LGBMClassifier(**best).fit(Xtr, y_all.iloc[tr], eval_set=[(Xva, y_all.iloc[va])], categorical_feature=CAT_COLS)
        oof[va] = m.predict_proba(Xva)[:, 1]; test_pred += m.predict_proba(Xte)[:, 1] / len(splits)
        aucs.append(roc_auc_score(y_all.iloc[va], oof[va]))
        print(f'Fold {k} AUC: {aucs[-1]:.5f}', flush=True)
    mask = ~np.isnan(oof)
    cv = roc_auc_score(y_all[mask], oof[mask])
    print(f'\nCV AUC (OOF): {cv:.5f} | vs 0.96072: {cv - 0.96072:+.5f}')
    if not SMOKE:
        np.save(OUT_DIR / 'oof_lgbm_tuned_final.npy', oof); np.save(OUT_DIR / 'test_lgbm_tuned_final.npy', test_pred)
        sub = pd.read_csv(DATA_DIR / 'sample_submission.csv'); sub['satisfaction'] = test_pred
        sub.to_csv(OUT_DIR / 'submission_lgbm_tuned_final.csv', index=False)
        print('saved: oof_lgbm_tuned_final.npy, test_lgbm_tuned_final.npy, submission_lgbm_tuned_final.csv')

## Summary & decisions
- **Result:** tuned LightGBM on `X_final` — 5-fold CV **0.96088** (+0.00016 vs the earlier 0.96072).
- **Interpretation:** tuning gave a small gain only. The big gains came from features (see `03`), and later from model *diversity* (`07`–`10`, TabPFN) rather than from squeezing one GBDT.
- **Role in the stack:** a strong, fast tree model with predictions different in character from neural networks.
- **Next:** `05` trains CatBoost and XGBoost on the same features and folds.